In [ ]:
import os
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, ResNet152, ResNet101
from tensorflow.keras.applications.resnet import preprocess_input
from skimage import exposure
from skimage.io import imread
from skimage.transform import resize
from skimage.color import rgb2gray
import pandas as pd
from skimage.filters import gabor_kernel
from scipy import ndimage as ndi
from scipy.fftpack import dct
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D
import pandas as pd
import os

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

# ==============================
#  TRY TENSORFLOW ViT FIRST
# ==============================
try:
    from tensorflow.keras.applications import vit
    import tensorflow as tf

    print("✅ Using TensorFlow ViT model")

    vit_model = vit.ViT_B16(
        include_top=False,
        pretrained="imagenet21k+imagenet2012",
        pooling="avg"
    )

    def extract_vit_features_tf(image_path):
        img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
        x = tf.keras.preprocessing.image.img_to_array(img)
        x = tf.expand_dims(x, axis=0)
        x = tf.keras.applications.resnet.preprocess_input(x)
        features = vit_model.predict(x, verbose=0)[0]
        return features

    use_huggingface = False

except Exception as e:
    print("⚠️ TensorFlow ViT not available, switching to HuggingFace ViT")
    print("Reason:", e)
    use_huggingface = True

# ==============================
#  FALLBACK → HUGGINGFACE ViT
# ==============================
if use_huggingface:
    import torch
    from transformers import ViTModel, ViTImageProcessor

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
    vit_model = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k").to(device)
    vit_model.eval()

    def extract_vit_features_hf(image_path):
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = vit_model(**inputs)
            features = outputs.pooler_output.squeeze().cpu().numpy()
        return features


# ===================================================
# === PART 2: UPDATE DATA READING LIKE YOUR EUPEA CODE
# ===================================================

# --- Define fold directories ---
base_dir = r"/kaggle/input/datasets/saurabhsainiji/anno-casewise-train-balanced-test-untouched/Anno-casewise-spilit-train-balanced-test-untouched_/Patient-wise/Split-3"
train_dir = os.path.join(base_dir, "Training")
test_dir = os.path.join(base_dir, "Testing")

labels = ['Category-1', 'Category-2', 'Category-3']


# ==============================
#  COMMON FEATURE EXTRACTION LOGIC
# ==============================
def extract_vit_features(image_path):
    """Automatically choose TF-ViT or HF-ViT"""
    if use_huggingface:
        return extract_vit_features_hf(image_path)
    else:
        return extract_vit_features_tf(image_path)


# ==============================
#  FUNCTION: Extract features from folder
# ==============================
def extract_features_from_directory(data_dir):
    df = pd.DataFrame(columns=['features', 'labels', 'image_name'])

    for label_idx, label in enumerate(labels):
        label_dir = os.path.join(data_dir, label)

        if not os.path.exists(label_dir):
            print(f"⚠️ Warning: {label_dir} not found")
            continue

        for subdir, dirs, files in os.walk(label_dir):
            for file in tqdm(files, desc=f"Extracting {label}"):

                if not (file.lower().endswith(".jpg") or file.lower().endswith(".png")):
                    continue

                img_path = os.path.join(subdir, file)

                try:
                    # ----- Extract ViT features -----
                    feats = extract_vit_features(img_path)

                    df.loc[len(df)] = [feats, label_idx, file]

                except Exception as e:
                    print(f"❌ Error processing {file}: {e}")

    return df


# ==============================
#  CREATE TRAIN + TEST FEATURES
# ==============================
vit_train_df = extract_features_from_directory(train_dir)
vit_test_df = extract_features_from_directory(test_dir)


# ==============================
#  RESULTS CHECK
# ==============================
print("\nTRAIN SET:")
print(vit_train_df.head())
print("\nTEST SET:")
print(vit_test_df.head())

print(f"\n🔍 Feature dimension example: {vit_train_df['features'][0].shape}")


In [ ]:
#print("BF-AD-AP-Resnet50 dataframe", res_50_train_features_df)

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, precision_score, f1_score, accuracy_score

# --- ✅ Separate features, labels, and image names for training and testing sets ---
X_train = np.stack(vit_train_df['features'].values)
y_train = np.array(vit_train_df['labels'].values)

X_test = np.stack(vit_test_df['features'].values)
y_test = np.array(vit_test_df['labels'].values)
image_names = np.array(vit_test_df['image_name'].values)  # store filenames

print("Vit features with SVM (Train/Test evaluation)")

# --- ✅ Initialize and Train SVM ---
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

# --- ✅ Predict on test data ---
y_pred = svm.predict(X_test)

# --- ✅ Compute confusion matrix ---
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", conf_matrix)

# --- ✅ Compute metrics ---
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
sensitivity = np.mean([
    conf_matrix[i, i] / conf_matrix[i, :].sum() if conf_matrix[i, :].sum() > 0 else 0
    for i in range(conf_matrix.shape[0])
])
specificity = np.mean([
    (np.sum(conf_matrix) - np.sum(conf_matrix[i, :]) - np.sum(conf_matrix[:, i]) + conf_matrix[i, i]) /
    (np.sum(conf_matrix) - np.sum(conf_matrix[:, i])) if np.sum(conf_matrix[:, i]) > 0 else 0
    for i in range(conf_matrix.shape[0])
])
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

# --- ✅ Print overall results ---
print(f"\nAccuracy: {accuracy:.6f}")
print(f"Specificity: {specificity:.6f}")
print(f"Sensitivity (Recall): {sensitivity:.6f}")
print(f"Precision: {precision:.6f}")
print(f"F1 Score: {f1:.6f}")

# --- ✅ Identify and print misclassified images ---
print("\nMisclassified Images:")
misclassified_indices = np.where(y_test != y_pred)[0]

if len(misclassified_indices) == 0:
    print("🎉 No misclassifications! All predictions are correct.")
else:
    for idx in misclassified_indices:
        true_label = labels[y_test[idx]]
        pred_label = labels[y_pred[idx]]
        img_name = image_names[idx]
        print(f"Image: {img_name} | True: {true_label} | Predicted: {pred_label}")
